In [ ]:
%load_ext autoreload
%autoreload 2
%xmode verbose

In [ ]:
from ExoRM import read_rm_data, load_model, ForecasterRM, preprocess_data

import numpy
import matplotlib.pyplot as plot
import matplotlib
import pandas
import seaborn
import math
import time

save = True # save the figures / csv to files
plot.style.use('seaborn-v0_8-paper')
seaborn.set_theme(style = 'white', context = 'paper')
matplotlib.rcParams['figure.figsize'] = [4, 3]
matplotlib.rcParams['axes.labelsize'] = 10  # Axis label font size
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['mathtext.fontset'] = 'cm'
matplotlib.rcParams['figure.dpi'] = 300
matplotlib.rcParams['figure.constrained_layout.use'] = True

matplotlib.rcParams['lines.markersize'] = 1.2  # Default marker size (scatter size)
matplotlib.rcParams['lines.linewidth'] = 2   # Default line width

path = 'Paper Material/ExoRM'
data_path = f'{path}/no_mass_planets.csv'
data = pandas.read_csv(data_path)
data = data.set_index('pl_name')

data

In [ ]:
x = data['pl_rade']
x = numpy.log10(x)

model = load_model()
y = model(x)
data['mass_est'] = 10 ** y

plot.scatter(x, y)
plot.xlabel('Radius (R$_{\\oplus}$)')
plot.ylabel('Mass (M$_{\\oplus}$)')
plot.title('Exoplanet Radius-Mass Distribution')

# if save: plot.savefig(f'{path}/Figure 1.jpeg')

plot.show()

In [ ]:
columns = ['pl_rade', 'mass_est']

save_data = data[columns].copy()
save_data[columns] = save_data[columns]

save_data[columns] = save_data[columns].map(
    lambda x: x if x == 0 or math.isnan(x) else round(x, (5 - 1) - int(math.floor(math.log10(abs(x)))))
)

if save: save_data.to_csv(f'{path}/ExoRM_predictions.csv')
save_data

In [ ]:
# from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive
#
# table = NasaExoplanetArchive.query_criteria(
#     table = 'PS',
#     select = 'pl_name, pl_bmasse, pl_rade, pl_radeerr1, pl_radeerr2',
#     where = '''soltype='Published Confirmed' AND pl_controv_flag = 0'''
# )
#
# data = table.to_pandas()
#
# no_mass = data.groupby('pl_name')['pl_bmasse'].apply(lambda x: x.isnull().all())
# data = data[data['pl_name'].isin(no_mass[no_mass].index)].copy()
#
# data['error_score'] = (
#     abs(data['pl_radeerr1'] / data['pl_rade']) +
#     abs(data['pl_radeerr2'] / data['pl_rade'])
# ) / 2
#
# data = data.dropna(subset = ['pl_rade', 'error_score'])
# data = data.loc[data.groupby('pl_name')['error_score'].idxmin()]
# data['name_len'] = data['pl_name'].str.len()
# data = data.sort_values(by = ['name_len', 'pl_name'], ascending = True)
# data = data.drop('name_len', axis = 1)
#
# data.to_csv(data_path, index = False)
#
# data